<a href="https://colab.research.google.com/github/UmarAamir6684/ECHO---Quantathon/blob/main/Quantathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip -q install qiskit qiskit-aer

# ECHO — Entangled Minds: The Quantathon
**Participant:** Umar Aamir Faridi  
**Team name:** myTeam  
**Stretch module selected:** 06 — The Impossible Claim

## Module 01 — The Gatekeeper (Mandatory)

### Bronze: always 1 and a 50/50 bit

In [4]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import pi, sqrt

simulator = AerSimulator()

def counts_for(circuit, shots=1000):
    measured = circuit.copy()
    if measured.num_clbits == 0:
        measured.measure_all()
    compiled = transpile(measured, simulator)
    return simulator.run(compiled, shots=shots).result().get_counts()

# Always-1 bit: X|0> = |1>
always_one = QuantumCircuit(1, 1)
always_one.x(0)
always_one.measure(0, 0)
counts_one = counts_for(always_one)
print("Always-1 bit:", counts_one)

# Fair quantum bit: H|0> gives equal probabilities for 0 and 1
fair_bit = QuantumCircuit(1, 1)
fair_bit.h(0)
fair_bit.measure(0, 0)
counts_fair = counts_for(fair_bit, shots=1000)
print("Fair quantum bit (1000 shots):", counts_fair)

Always-1 bit: {'1': 1000}
Fair quantum bit (1000 shots): {'0': 490, '1': 510}


**What it proves:** The X gate makes the measured bit 1 in an ideal simulation. The H gate on |0⟩ produces a 50/50 distribution in the ideal model; finite-shot counts can differ slightly from exactly equal.

### Silver: uncertain-looking but certain

In [5]:
# H followed by H is the identity: H(H|0>) = |0>
certain_after_two_gates = QuantumCircuit(1, 1)
certain_after_two_gates.h(0)
certain_after_two_gates.h(0)
certain_after_two_gates.measure(0, 0)
counts_hh = counts_for(certain_after_two_gates)
print("H then H:", counts_hh)

H then H: {'0': 1000}


**What it proves:** Although the first H creates a superposition, the second H reverses it. The final measurement is 0 in an ideal simulation.

### Gold: produce 101 from 000

In [6]:
# Qubits begin in 000. Flip q0 and q2. Qiskit displays classical bits as c2c1c0.
vault = QuantumCircuit(3, 3)
vault.x(0)
vault.x(2)
vault.measure([0, 1, 2], [0, 1, 2])
counts_101 = counts_for(vault)
print("Vault code:", counts_101)

Vault code: {'101': 1000}


**What it proves:** Applying X to the first and third qubits changes 000 to the measured pattern 101.

---

## Module 03 — The Two Observers (Mandatory)

### Bronze: create a pair that agrees

In [7]:
bell_pair = QuantumCircuit(2, 2)
bell_pair.h(0)
bell_pair.cx(0, 1)
bell_pair.measure([0, 1], [0, 1])
bell_counts = counts_for(bell_pair, shots=1000)
print("Bell-pair measurements:", bell_counts)

Bell-pair measurements: {'11': 518, '00': 482}


**What it proves:** The Bell state created by H followed by CNOT produces 00 and 11, with no 01 or 10 outcomes in an ideal computational-basis measurement.

### Silver: compare with a shared classical bit

In [8]:
# Classical comparison: both observers output the same pre-agreed random bit.
import random
classical_pair_counts = {"00": 0, "11": 0}
for _ in range(1000):
    shared_bit = random.choice([0, 1])
    classical_pair_counts[str(shared_bit) * 2] += 1
print("Shared-classical-bit comparison:", classical_pair_counts)

Shared-classical-bit comparison: {'00': 495, '11': 505}


**What it proves:** Perfect agreement by itself does not prove entanglement. Two observers could share a classical random bit in advance and always report the same result.

### Gold: test correlations in different bases (CHSH)

In [9]:
# CHSH test for a Bell pair. Each angle defines a measurement basis.
# Outcomes are mapped to +1 for bit 0 and -1 for bit 1.
# For |Phi+>, the expected correlations for these settings give |S| = 2*sqrt(2).

def bell_correlation(theta_a, theta_b, shots=4000):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    # Rotate each measurement basis before computational-basis measurement.
    qc.ry(-theta_a, 0)
    qc.ry(-theta_b, 1)
    qc.measure([0, 1], [0, 1])
    counts = counts_for(qc, shots=shots)
    total = sum(counts.values())
    correlation_sum = 0
    for bits, n in counts.items():
        # Qiskit count strings are c1c0: left is qubit 1, right is qubit 0.
        bit_b = int(bits[0])
        bit_a = int(bits[1])
        value_a = 1 if bit_a == 0 else -1
        value_b = 1 if bit_b == 0 else -1
        correlation_sum += value_a * value_b * n
    return correlation_sum / total, counts

A0, A1 = 0, pi/2
B0, B1 = pi/4, -pi/4

E00, c00 = bell_correlation(A0, B0)
E01, c01 = bell_correlation(A0, B1)
E10, c10 = bell_correlation(A1, B0)
E11, c11 = bell_correlation(A1, B1)
S = E00 + E01 + E10 - E11

print("E(A0,B0) =", round(E00, 3), "counts:", c00)
print("E(A0,B1) =", round(E01, 3), "counts:", c01)
print("E(A1,B0) =", round(E10, 3), "counts:", c10)
print("E(A1,B1) =", round(E11, 3), "counts:", c11)
print("CHSH S =", round(S, 3), "(ideal quantum magnitude =", round(2*sqrt(2), 3), ")")
print("Classical local-hidden-variable bound: |S| <= 2")

E(A0,B0) = 0.7 counts: {'01': 305, '11': 1718, '10': 296, '00': 1681}
E(A0,B1) = 0.684 counts: {'11': 1601, '10': 307, '00': 1767, '01': 325}
E(A1,B0) = 0.72 counts: {'01': 286, '10': 275, '00': 1697, '11': 1742}
E(A1,B1) = -0.703 counts: {'11': 322, '10': 1683, '00': 273, '01': 1722}
CHSH S = 2.806 (ideal quantum magnitude = 2.828 )
Classical local-hidden-variable bound: |S| <= 2


**What it proves:** In this simulated experiment, the Bell-pair correlations should exceed the classical CHSH bound of 2, approaching 2√2 as the number of shots increases. This is stronger evidence of nonclassical correlations than agreement in one measurement basis. It does **not** demonstrate faster-than-light messaging. The notebook uses an ideal simulator, not a loophole-free physical Bell test.

---

## Module 05 — The Damaged Message (Mandatory)

### Bronze: recover the original message by majority vote

In [10]:
triples = """100 011 001 001 001 011 100 110
001 011 000 000 001 100 011 011
000 111 000 000 110 000 001 000
000 111 001 010 101 011 111 111""".split()

recovered_bits = []
for triple in triples:
    ones = triple.count("1")
    recovered_bits.append("1" if ones >= 2 else "0")

bitstream = "".join(recovered_bits)
bytes_8 = [bitstream[i:i+8] for i in range(0, len(bitstream), 8)]
decoded_message = "".join(chr(int(byte, 2)) for byte in bytes_8)

print("Recovered bitstream:", bitstream)
print("8-bit groups:", bytes_8)
print("Decoded message:", decoded_message)

Recovered bitstream: 01000101010000110100100001001111
8-bit groups: ['01000101', '01000011', '01001000', '01001111']
Decoded message: ECHO


**What it proves:** Given the stated condition that at most one bit in each triple was flipped, majority vote recovers the original bits. The recovered ASCII message is ECHO.

### Silver: show when majority vote fails

In [11]:
examples = ["000", "001", "011", "110"]
for triple in examples:
    ones = triple.count("1")
    majority = "1" if ones >= 2 else "0"
    print("Triple:", triple, "| majority result:", majority)

Triple: 000 | majority result: 0
Triple: 001 | majority result: 0
Triple: 011 | majority result: 1
Triple: 110 | majority result: 1


**What it proves:** If two bits in a triple are corrupted, majority vote can return the wrong original bit. For example, an original 000 changed to 110 would be decoded as 1.

### Gold: correction versus detection; quantum no-cloning

A three-bit repetition code corrects **one bit flip per triple** by majority vote. If the three bits disagree, an error can often be noticed, but majority vote does not reliably correct two flips and cannot always identify every multi-bit error. An arbitrary unknown quantum state cannot be protected by simply copying it three times because the no-cloning theorem forbids perfect copying of an arbitrary unknown quantum state. Quantum error-correcting codes encode quantum information across multiple qubits without making independent copies of the unknown state.

**What it proves:** Classical repetition coding has a clear correction limit; quantum error correction must use a different strategy.

---

## Module 06 — The Impossible Claim (Selected Stretch Challenge)

### Bronze: teleport a quantum state

In [12]:
# Teleport the input state |1> from q0 to q2.
# q1 and q2 begin as the shared entangled resource.
teleport = QuantumCircuit(3, 3)
teleport.x(0)                 # input state |1>
teleport.h(1)
teleport.cx(1, 2)             # shared Bell pair on q1, q2
teleport.cx(0, 1)
teleport.h(0)
teleport.measure(0, 0)        # Alice's first classical bit
teleport.measure(1, 1)        # Alice's second classical bit

# Receiver's corrections controlled by the two classical measurement bits.
with teleport.if_test((teleport.clbits[1], 1)):
    teleport.x(2)
with teleport.if_test((teleport.clbits[0], 1)):
    teleport.z(2)

teleport.measure(2, 2)
teleport_counts = counts_for(teleport, shots=1000)
print("Teleportation circuit:")
print(teleport.draw())
print("Counts (c2c1c0):", teleport_counts)

Teleportation circuit:
     ┌───┐          ┌───┐┌─┐                                                   »
q_0: ┤ X ├───────■──┤ H ├┤M├───────────────────────────────────────────────────»
     ├───┤     ┌─┴─┐└┬─┬┘└╥┘                                                   »
q_1: ┤ H ├──■──┤ X ├─┤M├──╫────────────────────────────────────────────────────»
     └───┘┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  ┌───┐ ───────┐»
q_2: ─────┤ X ├───────╫───╫───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─┤ Z ├  End-0 ├»
          └───┘       ║   ║   └──╥───  └───┘ ───────┘   └──╥───  └───┘ ───────┘»
                      ║   ║ ┌────╨────┐               ┌────╨────┐              »
c: 3/═════════════════╩═══╩═╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞══════════════»
                      1   0 └─────────┘               └─────────┘              »
«         
«q_0: ────
«         
«q_1: ────
«      ┌─┐
«q_2: ─┤M├
«      └╥┘
«c: 3/══╩═
«       2 
Counts (c2c1c0): {'100': 236, '101': 249, '110': 250, '111': 265}


**What it proves:** When the protocol's classical measurement results are used for the receiver's corrections, the receiver's measured qubit should be 1 for this chosen input state in an ideal simulation. The two classical bits are part of the protocol.

### Silver: what if the classical bits do not arrive?

In [13]:
# Run the same preparation and Alice-side measurements, but omit receiver corrections.
# Before correction, q2's local measurement is random for this |1> input.
no_classical_corrections = QuantumCircuit(3, 3)
no_classical_corrections.x(0)
no_classical_corrections.h(1)
no_classical_corrections.cx(1, 2)
no_classical_corrections.cx(0, 1)
no_classical_corrections.h(0)
no_classical_corrections.measure(0, 0)
no_classical_corrections.measure(1, 1)
no_classical_corrections.measure(2, 2)

no_correction_counts = counts_for(no_classical_corrections, shots=1000)
receiver_zero = sum(n for bits, n in no_correction_counts.items() if bits[0] == "0")
receiver_one = sum(n for bits, n in no_correction_counts.items() if bits[0] == "1")
print("Counts without receiver corrections (c2c1c0):", no_correction_counts)
print("Receiver q2 measured 0:", receiver_zero)
print("Receiver q2 measured 1:", receiver_one)

Counts without receiver corrections (c2c1c0): {'100': 255, '010': 248, '011': 251, '101': 246}
Receiver q2 measured 0: 499
Receiver q2 measured 1: 501


**What it proves:** Without the classical information and corrections, the receiver's local measurement is random rather than a usable copy of the sender's chosen input. Finite-shot counts will fluctuate around 50/50.

### Gold: conclusion about faster-than-light messaging

The teleportation protocol uses a shared entangled pair plus two ordinary classical bits. The receiver cannot reconstruct the input state from entanglement alone; the classical results are necessary to choose the correction. Because usable information depends on classical communication, which cannot be sent faster than light, this protocol does not establish faster-than-light messaging.

**What it proves:** Quantum teleportation transfers a quantum state using entanglement and classical communication; it does not transmit a controllable message faster than light.

---

# ECHO Evidence Board

| FACT — directly observed or given | CLAIM — asserted by a person or log | INFERENCE — interpretation | OPEN QUESTION — not resolved by these experiments |
|---|---|---|---|
| Module 05's supplied 32 triples decode to the ASCII word ECHO under the stated one-flip-per-triple assumption. | Zara claimed ECHO sent a message faster than light. | The circuit demonstrations do not support faster-than-light messaging by entanglement alone. | Who configured ARIA and who, if anyone, caused Zara's disappearance? |
| In the simulator, the Bell pair gives matching computational-basis outcomes and the CHSH correlations should violate the classical bound. | ARIA stated that no unauthorised command was issued. | Agreement alone is not proof of entanglement; multiple measurement settings provide a stronger correlation test. | Were the observatory logs altered, and by whom? |
| The teleportation circuit requires two classical measurement bits for receiver corrections. | Sana said she deleted a corrupted log; Bilal said he had warned Zara; Professor Vance said ECHO was never real. | The scientific claim and the human disappearance are separate questions. The supplied evidence does not justify naming a culprit. | What do the original ECHO seed and recovered circuit show when run in the official notebook? |